# Finetune BERT (bert-base-uncased) for Next Sentence Prediction
**Solhee Tucker**  
May 2026  
Final project for ML in Bioinformatics class, 2026 spring, San José State University  
Instructor: Dr. Williams (Bill) Andreopoulos

### GPU Setup

In [1]:
# === GPU / device setup (works in Google Colab and local) ===
import sys, platform, torch

IN_COLAB = "google.colab" in sys.modules

print(f"Running in Google Colab : {IN_COLAB}")
print(f"Platform               : {platform.platform()}")
print(f"Python                 : {sys.version.split()[0]}")
print(f"PyTorch                : {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA available         : True")
    print(f"CUDA device name       : {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count      : {torch.cuda.device_count()}")
    print(f"CUDA version (torch)   : {torch.version.cuda}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("No GPU detected -- falling back to CPU")

print(f"\n>>> Using device: {device}")


Running in Google Colab : False
Platform               : Windows-10-10.0.26200-SP0
Python                 : 3.11.15
PyTorch                : 2.12.0.dev20260408+cu128
CUDA available         : True
CUDA device name       : NVIDIA GeForce RTX 5090
CUDA device count      : 1
CUDA version (torch)   : 12.8

>>> Using device: cuda


### Library Imports

We will use HuggingFace `transformers` (`BertTokenizer`, `BertForNextSentencePrediction`), PyTorch for training, and `scikit-learn` for accuracy. Seeds are fixed for reproducibility.

In [2]:
# If running in Colab for the first time, uncomment to install:
# !pip install -q transformers scikit-learn pandas matplotlib tqdm

import os
import random
import numpy as np
import pandas as pd

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# HuggingFace Transformers
import transformers
from transformers import (
    BertTokenizer,
    BertForNextSentencePrediction,
    get_linear_schedule_with_warmup,
)

# Metrics & utilities
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 120)
print("transformers :", transformers.__version__)
print("pandas       :", pd.__version__)
print("numpy        :", np.__version__)
print("All imports OK")


transformers : 5.8.1
pandas       : 3.0.3
numpy        : 2.4.4
All imports OK


### Load Datasets

The CSVs have **no header**, contain trailing empty columns (artifact of Excel export), and are encoded in `latin-1` (not UTF-8). We handle all three when loading.

**Column convention**: `sentence_a`, `sentence_b`, `label`  
**Label convention**: `0` = IsNextSentence (positive), `1` = NotNextSentence (negative).

In [3]:
from pathlib import Path

DATA_DIR = Path("../content")


def load_pairs(filename: str) -> pd.DataFrame:
    path = DATA_DIR / filename
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["sentence_a", "sentence_b", "label"],
        encoding="latin-1",
    )
    df["label"] = df["label"].astype(int)
    return df


test_df = load_pairs("TestSet-2.csv")
train_df = load_pairs("trainingdata-2.csv")
val_df = load_pairs("validationdata-2.csv")

print(">>> Dataset shapes (before deduplication)")
print(f"train : {train_df.shape}")
print(f"val   : {val_df.shape}")
print(f"test  : {test_df.shape}")

train_df = train_df.drop_duplicates().reset_index(drop=True)
val_df = val_df.drop_duplicates().reset_index(drop=True)
test_df = test_df.drop_duplicates().reset_index(drop=True)

print(">>> Dataset shapes (after deduplication)")
print(f"train : {train_df.shape}")
print(f"val   : {val_df.shape}")
print(f"test  : {test_df.shape}")


>>> Dataset shapes (before deduplication)
train : (960, 3)
val   : (137, 3)
test  : (103, 3)
>>> Dataset shapes (after deduplication)
train : (954, 3)
val   : (137, 3)
test  : (103, 3)


In [4]:
display(train_df.head())

,sentence_a,sentence_b,label
0,The Philippines is a country in Southeast Asia with a rich cultural heritage.,"Filipino cuisine includes dishes influenced by Chinese, Spanish, and American cuisine.",0
1,Singapore is a small island nation in Southeast Asia known for its modernity and cleanliness.,"The Merlion, a mythical creature with the head of a lion and the body of a fish, is a symbol of Singapore.",0
2,South Korea is a country in East Asia known for its K-pop music and popular dramas.,"Gangnam Style, a song by the South Korean singer Psy, became a global hit in 2012.",0
3,Sri Lanka is an island nation in South Asia known for its tea plantations and ancient temples.,Colombo is the largest city in Sri Lanka.,0
4,Taiwan is a democratic country in East Asia with a thriving economy.,"Taipei 101, a skyscraper in Taipei, was the tallest building in the world from 2004 to 2010.",0


In [5]:

print("\n>>> Label distribution (0 = IsNext, 1 = NotNext)")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df["label"].value_counts().to_dict()
    print(f"  {name:5s}: {counts}  (total={len(df)})")

# Sentence-length statistics
word_counts = pd.concat([
    df[col].str.split().str.len()
    for df in [train_df, val_df, test_df]
    for col in ["sentence_a", "sentence_b"]
])
print(word_counts.describe(percentiles=[0.5, 0.95, 0.99]))



>>> Label distribution (0 = IsNext, 1 = NotNext)
  train: {1: 480, 0: 474}  (total=954)
  val  : {1: 71, 0: 66}  (total=137)
  test : {0: 54, 1: 49}  (total=103)
count    2388.000000
mean       11.405360
std         3.714554
min         6.000000
50%        11.000000
95%        18.000000
99%        21.130000
max        24.000000
dtype: float64


In [6]:
# Source: https://huggingface.co/docs/transformers/v4.22.1/en/model_doc/bert#transformers.BertForNextSentencePrediction.forward.example
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased").to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Source: https://huggingface.co/docs/transformers/v4.22.1/en/model_doc/bert#transformers.BertForNextSentencePrediction.forward.example
prompt = "In Italy, pizza served in formal settings, such as at a restaurant, is presented unsliced."
next_sentence = "The sky is blue due to the shorter wavelength of blue light."
encoding = tokenizer(prompt, next_sentence, return_tensors="pt").to(device)

outputs = model(**encoding, labels=torch.LongTensor([1]).to(device))
logits = outputs.logits
assert logits[0, 0] < logits[0, 1]  # next sentence was random

In [8]:
""  # one row
row = train_df.iloc[0]
print("sentence_a :", row["sentence_a"])
print("sentence_b :", row["sentence_b"])
print("label      :", row["label"])


sentence_a : The Philippines is a country in Southeast Asia with a rich cultural heritage.
sentence_b : Filipino cuisine includes dishes influenced by Chinese, Spanish, and American cuisine.
label      : 0


In [9]:
MAX_LENG = 64
enc = tokenizer(
    row["sentence_a"],
    row["sentence_b"],
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENG,
).to(device)

In [10]:
for k, v in enc.items():
    print(f"\t{k:15s} : {v.shape}  (dtype={v.dtype})")
print("\tdecoded:", tokenizer.decode(enc["input_ids"][0]))

model.eval()
with torch.no_grad():
    out = model(**enc)

print("\nlogits:", out.logits)  # shape (1, 2)
pred = torch.argmax(out.logits, dim=-1).item()
print(f"pred={pred}, true ={row['label']}")

	input_ids       : torch.Size([1, 31])  (dtype=torch.int64)
	token_type_ids  : torch.Size([1, 31])  (dtype=torch.int64)
	attention_mask  : torch.Size([1, 31])  (dtype=torch.int64)
	decoded: [CLS] the philippines is a country in southeast asia with a rich cultural heritage. [SEP] filipino cuisine includes dishes influenced by chinese, spanish, and american cuisine. [SEP]

logits: tensor([[ 6.3202, -6.1898]], device='cuda:0')
pred=0, true =0


In [11]:
preds = []
labels = []

model.eval()
with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Baseline evaluation"):
        enc = tokenizer(
            row["sentence_a"],
            row["sentence_b"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENG,
        ).to(device)

        logits = model(**enc).logits
        pred = torch.argmax(logits, dim=-1).item()

        preds.append(int(pred))
        labels.append(int(row["label"]))

correct = sum(p == l for p, l in zip(preds, labels))
baseline_acc = correct / len(labels)
print(f"\nBaseline accuracy: {baseline_acc:.4f}")


Baseline evaluation:   0%|          | 0/103 [00:00<?, ?it/s]


Baseline accuracy: 0.5922


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(labels, preds, target_names=["IsNext (0)", "NotNext (1)"], digits=4))
print("Confusion matrix:")
print(confusion_matrix(labels, preds))

              precision    recall  f1-score   support

  IsNext (0)     0.5638    0.9815    0.7162        54
 NotNext (1)     0.8889    0.1633    0.2759        49

    accuracy                         0.5922       103
   macro avg     0.7264    0.5724    0.4960       103
weighted avg     0.7185    0.5922    0.5067       103

Confusion matrix:
[[53  1]
 [41  8]]


## Finetuning </br>
Source: https://github.com/jamescalam/transformers/blob/main/course/training/06_nsp_training.ipynb

In [13]:
sentence_a = train_df["sentence_a"].tolist()
sentence_b = train_df["sentence_b"].tolist()
label = train_df["label"].tolist()

In [14]:
inputs = tokenizer(
    sentence_a,
    sentence_b,
    return_tensors='pt',
    max_length=MAX_LENG,
    truncation=True,
    padding='max_length'
)
inputs['labels'] = torch.LongTensor([label]).T

In [15]:
class MeditationsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

In [16]:
# Initialize our data using the MeditationDataset class.
dataset = MeditationsDataset(inputs)

loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

model.to(device)

BertForNextSentencePrediction(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [17]:
L_RATE = 2e-5

In [18]:
# activate training mode
model.train()
# initialize optimizer
optim = AdamW(model.parameters(), lr=L_RATE)

In [19]:
# Prep Validation
val_sentence_a = val_df["sentence_a"].tolist()
val_sentence_b = val_df["sentence_b"].tolist()
val_label = val_df["label"].tolist()

val_inputs = tokenizer(
    val_sentence_a, val_sentence_b,
    return_tensors='pt',
    max_length=MAX_LENG,
    truncation=True,
    padding='max_length',
)
val_inputs['labels'] = torch.LongTensor([val_label]).T

val_dataset = MeditationsDataset(val_inputs)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"val batches: {len(val_loader)}")

val batches: 9


In [20]:
epochs = 3

train_losses = []
val_losses = []
val_accs = []

for epoch in range(epochs):
    #================= Training =================
    model.train()
    running_loss = 0.0
    n_samples = 0
    loop = tqdm(loader, leave=True, desc=f"Epoch {epoch} [train]")

    # setup loop with TQDM and dataloader
    for batch in loop:
        # initialize calculated gradients (from prev step)
        optim.zero_grad()
        # pull all tensor batches required for training
        input_ids       = batch['input_ids'].to(device)
        attention_mask  = batch['attention_mask'].to(device)
        token_type_ids  = batch['token_type_ids'].to(device)
        labels          = batch['labels'].to(device)
        # process
        outputs = model(input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids,
                        labels=labels)
        # extract loss
        loss = outputs.loss
        # calculate loss for every parameter that needs grad update
        loss.backward()
        # update parameters
        optim.step()
        # print relevant info to progress bar
        running_loss += loss.item() * input_ids.size(0)
        n_samples += input_ids.size(0)

        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())
    avg_train_loss = running_loss / n_samples

    #================= Validation =================
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, leave=False, desc=f"Epoch {epoch} [val]"):
            input_ids       = batch['input_ids'].to(device)
            attention_mask  = batch['attention_mask'].to(device)
            token_type_ids  = batch['token_type_ids'].to(device)
            labels          = batch['labels'].to(device)

            outputs = model(input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids,
                            labels=labels)
            val_running_loss += outputs.loss.item() * input_ids.size(0)

            preds = torch.argmax(outputs.logits, dim=-1)  # (B,)
            val_correct += (preds == labels.view(-1)).sum().item()
            val_total += labels.size(0)

    avg_val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch}  |  train_loss={avg_train_loss:.4f}  "
          f"|  val_loss={avg_val_loss:.4f}  |  val_acc={val_acc:.4f}\n")

Epoch 0 [train]:   0%|          | 0/60 [00:00<?, ?it/s]

C:\Users\10120\AppData\Local\Temp\ipykernel_17476\2305117189.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 0 [val]:   0%|          | 0/9 [00:00<?, ?it/s]


Epoch 0  |  train_loss=0.1060  |  val_loss=0.0096  |  val_acc=0.9927



Epoch 1 [train]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/9 [00:00<?, ?it/s]


Epoch 1  |  train_loss=0.0003  |  val_loss=0.0001  |  val_acc=1.0000



Epoch 2 [train]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 2 [val]:   0%|          | 0/9 [00:00<?, ?it/s]


Epoch 2  |  train_loss=0.0001  |  val_loss=0.0001  |  val_acc=1.0000



## Test

In [21]:
ft_preds = []
ft_labels = []

model.eval()
with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc="Fine-tuned eval on test"):
        enc = tokenizer(
            row["sentence_a"],
            row["sentence_b"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENG,
        ).to(device)
        logits = model(**enc).logits
        pred = torch.argmax(logits, dim=-1).item()
        ft_preds.append(int(pred))
        ft_labels.append(int(row["label"]))

correct = sum(p == l for p, l in zip(ft_preds, ft_labels))
finetuned_acc = correct / len(ft_labels)

print(f"\n>>> Baseline test accuracy   : {baseline_acc:.4f}")
print(f">>> Fine-tuned test accuracy : {finetuned_acc:.4f}")
print(f">>> Improvement              : +{(finetuned_acc - baseline_acc)*100:.2f}%pt\n")

print("Classification report (fine-tuned):")
print(classification_report(ft_labels, ft_preds,
                            target_names=["IsNext (0)", "NotNext (1)"], digits=4))
print("Confusion matrix:")
print(confusion_matrix(ft_labels, ft_preds))

Fine-tuned eval on test:   0%|          | 0/103 [00:00<?, ?it/s]


>>> Baseline test accuracy   : 0.5922
>>> Fine-tuned test accuracy : 1.0000
>>> Improvement              : +40.78%pt

Classification report (fine-tuned):
              precision    recall  f1-score   support

  IsNext (0)     1.0000    1.0000    1.0000        54
 NotNext (1)     1.0000    1.0000    1.0000        49

    accuracy                         1.0000       103
   macro avg     1.0000    1.0000    1.0000       103
weighted avg     1.0000    1.0000    1.0000       103

Confusion matrix:
[[54  0]
 [ 0 49]]


## Results and Discussion

Before finetuning, the Baseline BERT model accuracy was  **0.5922**. </br>
Here's the Confusion matrix for the test set: </br>
Confusion matrix: </br>
[[53  1] </br>
 [41  8]] </br>
  </br>
After finetuning for 3 epochs, the accuracy improved to **1.0000**. </br>
Here's the Confusion matrix for the test set: </br>
Confusion matrix: </br>
Confusion matrix: </br>
[[54  0] </br>
 [ 0 49]] </br>

I think finetuning improved the model's accuracy dramatically because 1) This is a type of job the BERT model is designed for. 2) Relationships between sentence A and sentence B are very specific and clear.

Also, we can see finetuning is very effective way to improve models for tasks like this.

Reference: </br>
[1] https://medium.com/data-science/bert-for-next-sentence-prediction-466b67f8226f</br>
[2] https://medium.com/data-science/how-to-fine-tune-bert-with-nsp-8b5615468e12 </br>
[3] https://huggingface.co/docs/transformers/v4.22.1/en/model_doc/bert#transformers</br>
[4] I worked on this with Claude AI agnet, Anthropic.</br>
(I tried to share the dialog, however, Claude doesn't support sharing co-work sessions at this moment.)